# Employee Performance Predictor — Walkthrough Notebook

This notebook walks through the same pipeline as `main.py`, one step at a time, so you can see and explain each output.

> Run `python main.py` once first (or just run all cells — Step 1 generates the data).

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))   # so `src` can be imported
import pandas as pd
from src import config, data_generator, preprocessing, eda, train, evaluate, predict
pd.set_option('display.max_columns', 30)

## Step 1 — Simulate the company (synthetic HR data)

In [ ]:
raw = data_generator.add_real_world_noise(data_generator.generate_employee_data())
raw.head()

In [ ]:
print(raw.shape)
raw.isna().sum()

## Step 2 — Clean the data
Duplicates removed, text fixed, impossible values (age 999, 9999 training hours) set to missing, then missing values filled with median / mode.

In [ ]:
clean = preprocessing.clean_data(raw)
clean.describe().T

## Step 3 — Feature engineering

In [ ]:
df = preprocessing.engineer_features(clean)
df[['salary_per_level','projects_per_year','training_per_year','overtime_hours']].describe()

## Step 4 — Exploratory Data Analysis

In [ ]:
df['performance_category'].value_counts(normalize=True)

In [ ]:
df.groupby('performance_category')[['training_hours','peer_feedback_score','absences','job_satisfaction']].mean().loc[config.CLASS_ORDER]

In [ ]:
from IPython.display import Image
config.make_dirs()
eda.plot_drivers(df)
Image(str(config.IMAGES_DIR / '04_performance_drivers.png'))

## Step 5 — Train and compare models

In [ ]:
X_train, X_test, y_train, y_test = train.split_data(df)
results = train.compare_models(X_train, y_train)
results

In [ ]:
pipe = train.train_best(results, X_train, y_train)

## Step 6 — Evaluate on unseen test data

In [ ]:
metrics = evaluate.evaluate(pipe, X_test, y_test, results.loc[0, 'model'])
Image(str(config.IMAGES_DIR / '07_confusion_matrix.png'))

In [ ]:
fi = evaluate.plot_feature_importance(pipe, X_test, y_test)
fi.head(10)

## Step 7 — Predict for new employees

In [ ]:
new = pd.read_csv(config.NEW_EMPLOYEES_PATH) if config.NEW_EMPLOYEES_PATH.exists() else data_generator.generate_new_employees()
result = predict.predict_employees(new)
result[['employee_id','department','job_level','predicted_category','confidence','hr_recommendation']]